# Example run

In [ ]:
############################################################################
# this example script runs the ParamEstimator_SpaceAI code 
# for a single point and stores the output as a pickle object
############################################################################

Simple analysis for a binary system GW signal.
All GW parameters are required to inputed, then ParamEstimator_SpaceAI.py is used to reconstruct the signal and estimate the parameters and precision

## Initial settings

### Packages

In [2]:

######################################################
# load python packages
import pickle
import numpy as np
import time
import os

from Analysis_Scripts import constants as const
from Analysis_Scripts import WD

######################################################
# load parameter estimation code
import ParamEstimator_SpaceAI as AI_PE


### Set GW input

In [3]:

######################################################
# set parameters
### These parameters can be changed in order to evaluate the signal of various sources

#######################
# GW source
source_Mc = 0.75 # detector-frame chirp mass [Msol]])
source_q = 1.05 # mass ratio (m_1/m_2)
source_dL = 25. # luminosity distance [Mpc]
source_tau0= 0. #decided to be the same as coalescence time
source_tc = 0. # time of merger, measured from the solar equinox [seconds]
source_phi0 = 0. # [0,2*np.pi] phase of the GW signal at fGWmin
source_iota = np.pi/4. # [0,np.pi]angle between sky localization and orbital momentum of the binary [rad]
source_psi = np.pi*3/4. # [0,2*np.pi]polarization angle [rad]
source_RA = np.pi # [0,2*np.pi]source right ascesnsion [rad], max at np.pi/2 for other values maximised
source_DEC_title = -np.pi/6. # [-np.pi/2,np.pi/2] source declination [rad], max at np.pi/2
source_DEC = source_DEC_title - np.pi*23.4/180
duration = 1. # duration of the observation [years]

#######################
# detector
detector_fGWmin = 0.01 # smallest frequency considered [Hz]
detector_fGWmax = 4. # largest frequency considered [Hz]
detector_max_measurement_time = duration*const.year # if detector_max_measurement_time [seconds] is longer than the lifetime of the source in the band, only the last detector_max_measurement_time piece of the signal in the band is considered
detector_orbit_R_cb = 8.44e6 # radius of the orbit of the center of the baseline around Earth [m]
detector_orbit_R_sat = 2e7 # radius of the orbit of the satellite around Earth [m], for period
detector_orbit_t0 = const.year/4 # reference time for fixing orbit of satellite around Earth [seconds]
detector_orbit_RA0 = np.pi/2 # right ascension of the satellite at t0 [rad]
detector_orbit_DEC0 = -23.4*np.pi/180  # declination of the satellite at t0 [rad]

#######################
# post-newtonian order
PN_order_phase = 3.5
PN_order_amplitude = 3.0

#######################
# use fast tau evaluation?
fast_tau_evaluation = False # boolean Flag. Set to True to use leading-order
                           # expression for calculation of tau(f_GW), \
                           # if False the PN expressions will be used


### Initialization of the output folder

In [4]:
#######################
# set up path to store results
timestamp_str = time.strftime("%Y%m%d")
fpath_out = "ExampleOutput/results_Example_SpaceAI_"+timestamp_str+"/"
######################
# create output directory
if not os.path.exists(fpath_out):
    os.makedirs(fpath_out)

######################################################
# run
######################################################
# get some derived parameters
detector_orbit_period = AI_PE.antennaFuns_satellites.period_satellite_earth(detector_orbit_R_sat)


### Initialization and information of the studied binary (WDB)

In [5]:
#######################
# check type of binary
WDbinary = WD.WDclass(
        source_Mc=source_Mc,
        source_q=source_q,
        detector_fGWmin=detector_fGWmin,
        detector_fGWmax=detector_fGWmax,
        max_measurement_time=duration)

WDbinary.INFO()


-------------------------------------------
Type of binary:  WD-WD
The masses of the two bodies is  0.841  and  0.883  Solar Masses
Cutoff Frequency due to RLOF:  0.0641  Hz
Evaluation for frequencies between  0.0641  and  0.0641  Hz
Measurement duration is  3.17e-08  yr
Multiple critical point could appear at:
- From Earth rotation around the Sun from frequency  0.0214  Hz  and above
- WARNING:
  From Satellite rotation around Earth from frequency  0.173  Hz and above
-------------------------------------------



## Run

### Save inputted setting for future reference

In [6]:

#######################
# write info file
fo = open(fpath_out+'/info.txt', 'w')
fo.write('# RUNNING WITH AIMforGW_frequencies.ParamEstimator_SpaceAI\n')

fo.write('# inputs for GW source\n')
fo.write(str(source_Mc)+' # detector-frame chirp mass [Msol]]\n')
fo.write(str(source_q)+' # source_q, mass ratio (m_1/m_2)\n')
fo.write(str(source_dL)+' # source_dL, luminosity distance [Mpc]\n')
fo.write(str(source_phi0)+' # source_phi0, phase of the GW signal at fGWmin\n')
fo.write(str(source_tc)+' # source_tc, time of merger, measured from the solar equinox [seconds]\n')
fo.write(str(source_iota)+' # source_iota, angle between sky localization and orbital momentum of the binary [rad]\n')
fo.write(str(source_psi)+' # source_psi, polarization angle [rad]\n')
fo.write(str(source_RA)+' # source right ascension [rad]\n')
fo.write(str(source_DEC)+' # source declination [rad]\n')

fo.write('# inputs for detector\n')
fo.write(str(detector_fGWmin)+' # detector_fGWmin, smallest frequency considered [Hz]\n')
fo.write(str(detector_fGWmax)+' # detector_fGWmax, largest frequency considered [Hz]\n')
fo.write(str(detector_max_measurement_time)+' # detector_max_measurement_time, max length of signal considered [seconds]\n')
fo.write(str(detector_orbit_R_cb)+' # detector_orbit_R_cb, radius of the orbit of the center of the baseline around Earth [m]\n')
fo.write(str(detector_orbit_R_sat)+' # detector_orbit_R_sat, radius of the orbit of the satellite around Earth [m], for period\n')
fo.write(str(detector_orbit_t0)+' # detector_orbit_t0, reference time for fixing orbit of satellite around Earth [seconds]\n')
fo.write(str(detector_orbit_RA0)+' # detector_orbit_RA0, right ascension of the satellite at t0 [rad]\n')
fo.write(str(detector_orbit_DEC0)+' # detector_orbit_DEC0, declination of the satellite at t0 [rad]\n')

fo.write('# inputs computation\n')
fo.write(str(PN_order_phase)+' # PN_order_phase\n')
fo.write(str(PN_order_amplitude)+' # PN_order_amplitude\n')
fo.write(str(fast_tau_evaluation)+ ' # Fast tau(fGW) evaluation flag\n')
fo.close()

### Initialize the GW signal class

In [7]:

#######################
# create an instance of the GW_event class
event = AI_PE.GW_event(
   source_Mc,
   source_q,
   source_iota,
   source_phi0,
   source_tc,
   source_dL,
   source_RA,
   source_DEC,
   source_psi,
   detector_fGWmin,
   detector_fGWmax,
   detector_orbit_R_cb,
   detector_orbit_period,
   detector_orbit_t0,
   detector_orbit_RA0,
   detector_orbit_DEC0,
   PN_order_phase,
   PN_order_amplitude,
   fast_tau_evaluation = fast_tau_evaluation,
   verbose = True
   )

### Run the parameter reconstruction

In [ ]:
# compute Fisher Matrix
event.get_SNR_FisherMatrix()

### Extract the information and save them

In [8]:

# compute covariance matrices without priors
event.get_CoVaMat()
event.get_CoVaMat_dimless()
event.get_angular_resolution()
# compute covariance matrices with priors
event.get_CoVaMat_priors()
event.get_CoVaMat_priors_dimless()
event.get_angular_resolution_priors()
# save event
with open(fpath_out+'/data.pkl', 'wb') as outp:
    pickle.dump(event, outp, pickle.HIGHEST_PROTOCOL)

Starting calculation of FisherMat.
Finished calculation of SNR and FisherMat. Time elapsed: 695.2 sec


The output of the run is saved in the folder ExampleOutput/result_Example_SpaceAI_'date of the run' under the name data.pkl

## Output

### Print the information of interest

In [9]:
print(event.SNR2)
print(event.FisherMat)
print(event.CoVaMat)
print(event.CoVaMat_dimless)

170.240695573928
[[ 2.23009563e+17 -4.13173658e+11  6.06207728e+07  9.85863257e+09
   1.38138285e+10 -8.52518396e+00 -5.01947099e+11  6.15158963e+12
   9.78243249e+09]
 [-4.13173658e+11  7.69837516e+05 -1.07261739e+02 -1.74627405e+04
  -2.08548405e+04 -1.33101610e-05  8.61694773e+05 -7.48593592e+06
  -1.73277223e+04]
 [ 6.06207728e+07 -1.07261739e+02  1.15397298e+02  4.20696683e+00
   1.07839618e+01  5.49338629e+00 -5.97870341e+02  7.05137555e+03
   2.97471878e+00]
 [ 9.85863257e+09 -1.74627405e+04  4.20696683e+00  6.80988527e+02
   1.71880081e+03 -1.88210913e-13 -9.21690706e+04  1.10902167e+06
   6.75746708e+02]
 [ 1.38138285e+10 -2.08548405e+04  1.07839618e+01  1.71880081e+03
   6.97316006e+03  9.03660615e-07 -2.36942856e+05  5.18154567e+06
   1.70560550e+03]
 [-8.52518396e+00 -1.33101610e-05  5.49338629e+00 -1.88210913e-13
   9.03660615e-07  2.72385113e-01 -1.23873384e+00  2.85235500e+00
  -5.94943756e-02]
 [-5.01947099e+11  8.61694773e+05 -5.97870341e+02 -9.21690706e+04
  -2.369428

End